# Generación del conjunto de datos de entrenamiento (Churn)

El objetivo de esta libreta es construir el conjunto de datos de entrenamiento para un modelo de predicción de churn en Telco.

Se combinan:
- gold_churn_spine
- gold_customer_profile
- gold_customer_aggregations

El resultado se guarda como:
👉 gold_churn_training_dataset

In [ ]:
%pip install databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

In [ ]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from pyspark.sql.functions import col, count, when, max, round

In [ ]:
catalog = "workspace"
database = "telco_churn"

gold_spine_table = f"{catalog}.{database}.gold_churn_spine"
gold_customer_profile_table = f"{catalog}.{database}.gold_customer_profile"
gold_customer_aggregations_table = f"{catalog}.{database}.gold_customer_aggregations"

gold_training_dataset_table = f"{catalog}.{database}.gold_churn_training_dataset"

fe = FeatureEngineeringClient()

In [ ]:
spine_df = spark.table(gold_spine_table)

print(f"Rows: {spine_df.count():,}")
print(f"Columns: {len(spine_df.columns)}")

spine_df.printSchema()

In [ ]:
entity_key = "customer_id"
timestamp_key = "usage_event_time"

profile_feature_names = [
    "age","gender","contract_type","region","region_type",
    "tariff_plan","monthly_fee","num_lines","device_type",
    "acquisition_channel","payment_method","has_tv_bundle",
    "has_fiber","has_roaming","paperless_billing","autopay",
    "nps_score_at_start","is_active","age_group","contract_risk_group",
    "signup_date"
]

profile_lookup = FeatureLookup(
    table_name=gold_customer_profile_table,
    feature_names=profile_feature_names,
    lookup_key=entity_key,
    timestamp_lookup_key=timestamp_key
)

aggregation_feature_names = [
    "data_consumed_gb","call_minutes","bill_amount",
    "days_payment_late","nps_score","coverage_score",
    "bill_vs_data_ratio"
]

aggregations_lookup = FeatureLookup(
    table_name=gold_customer_aggregations_table,
    feature_names=aggregation_feature_names,
    lookup_key=entity_key,
    timestamp_lookup_key=timestamp_key
)

feature_lookups = [profile_lookup, aggregations_lookup]

In [ ]:
label = "label_will_churn"
exclude_columns = ["churn_date"]


training_dataset = fe.create_training_set(
    df=spine_df,
    feature_lookups=feature_lookups,
    label=label,
    exclude_columns=exclude_columns
)

In [ ]:
training_df = training_dataset.load_df()

print(f"Rows: {training_df.count():,}")
print(f"Columns: {len(training_df.columns)}")

training_df.printSchema()

In [ ]:
training_df.limit(5).toPandas()

In [ ]:
feature_columns = [
    c for c in training_df.columns
    if c not in ["customer_id", "window_end", "label_will_churn"]
]

nulls = training_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in feature_columns
]).collect()[0]

for c in feature_columns:
    print(f"{c}: {nulls[c]}")

In [ ]:
print("Spine:", spine_df.count())
print("Training:", training_df.count())

In [ ]:
total = training_df.count()

balance = (
    training_df.groupBy("label_will_churn")
    .count()
    .withColumn("pct", round(col("count") / total * 100, 2))
    .collect()
)

for r in balance:
    print(r)

In [ ]:
clean_training_df = training_df.filter("label_will_churn IS NOT NULL")

print(f"Final rows: {clean_training_df.count():,}")

In [ ]:
(
    clean_training_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_training_dataset_table)
)

print(f"Saved to {gold_training_dataset_table}")